# Export one training checkpoint -- qwen3.5-4b distillation (Colab)

Exports a single intermediate checkpoint of a `distill_train.ipynb` run (a
`checkpoint-<step>` folder on Google Drive) as merged bf16 safetensors to its OWN
revision of `carev01/qwen35-4b-graphrag`. This needs no training and no dataset build.

**Why it exists (2026-10-21).** The v2 run's val loss bottomed at step 100 (0.114) and
then rose while train loss fell to 0.014: the final model overfit. End to end it wrote
about twice the unsupported facts of today's cheap tier
(`docs/superpowers/v2-ingest-trial-2026-10-20.md`). `checkpoint-100` tests whether
stopping early fixes that.

**What it does:**
1. Finds the checkpoint on Drive and checks that it is complete.
2. Reads its own `trainer_state.json` to show the val loss at that step.
3. Reloads base + adapter, smoke-tests three held-out prompts for JSON, then pushes.

It **never overwrites** an existing revision that already holds weights, and it **never
modifies** the Drive checkpoint: the checkpoint is copied to local disk first.

**Runtime:** any GPU runtime with about 16 GB VRAM or more (T4 is slow but works; L4 or
A100 is faster). Merging needs about 30 GB of free local disk. Expect 15-30 minutes,
plus about 20 more if `PUSH_GGUF` is on.

**Use:** set the HF token secret (section 2), check section 3's values, then
**Runtime -> Run all**.

## 1. Install packages

Same install as `distill_train.ipynb` section 1.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def pip_install(*args: str) -> None:
    """Run pip, print ALL of its output, and raise on failure (never hidden)."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout, flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"pip install failed (exit {proc.returncode}): {' '.join(args)}")


if importlib.util.find_spec("unsloth") is None:
    pip_install("--upgrade", "pip")
    pip_install("unsloth")
    pip_install("trl", "datasets", "huggingface_hub")

for pkg in ("unsloth", "trl", "transformers", "peft", "datasets", "huggingface_hub"):
    print(f"{pkg}=={importlib.metadata.version(pkg)}")
print("packages ready")

## 2. HF token -- Colab secret or environment, never hardcoded

Set a Colab secret named `HF_TOKEN` (the key icon in the left sidebar) *or* export
`HF_TOKEN` in the environment before starting this runtime. This cell never prints the
token -- only whether one was found.

In [ ]:
import os


def get_hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok
        except Exception:
            pass  # secret not set -- fall through to the environment
    except ImportError:
        pass  # not running on Colab
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        raise RuntimeError(
            "no HF token found -- add a Colab secret named HF_TOKEN, or set the "
            "HF_TOKEN environment variable, before running this cell")
    return tok


HF_TOKEN = get_hf_token()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF token loaded and logged in (value never printed)")

## 3. Configuration

`CHECKPOINT_STEP` picks the folder; `HF_MODEL_REVISION` is the branch it is pushed to.
`DRIVE_RUN_DIR` must match the training run's `RUN_NAME` (v2: `opus-distill-v2`).

In [ ]:
from pathlib import Path

# --- which checkpoint ------------------------------------------------------------------
RUN_NAME = "opus-distill-v2"                 # the training run's RUN_NAME
CHECKPOINT_STEP = 100                        # val loss bottomed here in the v2 run
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/graph-rag-distill") / RUN_NAME
CKPT_DIR = DRIVE_RUN_DIR / "checkpoints"
CHECKPOINT_DIR = CKPT_DIR / f"checkpoint-{CHECKPOINT_STEP}"
FINAL_ADAPTER_DIR = DRIVE_RUN_DIR / "final-adapter"   # tokenizer fallback only

# --- held-out prompts for the smoke test (the same val set the run used) --------------
HF_DATASET_REPO = "carev01/ft-opus-distill"
HF_DATASET_REVISION = "v2"

# --- model: must match the training run --------------------------------------------
MAX_SEQ_LENGTH = 20480
LOAD_IN_4BIT = True    # how the adapter was trained; the merge below writes bf16 anyway

# --- export -----------------------------------------------------------------------------
HF_MODEL_REPO = "carev01/qwen35-4b-graphrag"
HF_MODEL_REVISION = f"opus-distill-2026-10-ckpt{CHECKPOINT_STEP}"
HF_MODEL_PRIVATE = False   # the repo is public by design (Vast pulls it without a token)
PUSH_GGUF = False          # vLLM serves the safetensors; turn on only for llama.cpp use

LOCAL_CKPT = Path(f"/content/ckpt/checkpoint-{CHECKPOINT_STEP}")
OUT_DIR = Path(f"/content/outputs/{HF_MODEL_REVISION}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{CHECKPOINT_DIR} -> {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

## 4. Mount Drive and check the checkpoint

A checkpoint restored from the Drive trash goes back to its original folder. The cell
lists every checkpoint it finds, so a wrong `CHECKPOINT_STEP` or `RUN_NAME` shows at once.
The checkpoint is then copied to local disk; tokenizer files are added from the final
adapter only when the checkpoint lacks them. Drive is never written to.

In [ ]:
import json
import shutil

from google.colab import drive

drive.mount("/content/drive")
_found = sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print(f"checkpoints under {CKPT_DIR}: {[p.name for p in _found] or 'none'}")
assert CHECKPOINT_DIR.is_dir(), (
    f"{CHECKPOINT_DIR} not found -- restore it from the Drive trash (it goes back to "
    "this folder), or fix RUN_NAME / CHECKPOINT_STEP")
assert (CHECKPOINT_DIR / "adapter_config.json").exists() and any(
    CHECKPOINT_DIR.glob("adapter_model.*")), (
    f"{CHECKPOINT_DIR} has no adapter_config.json + adapter_model.* -- incomplete checkpoint")

if LOCAL_CKPT.exists():
    shutil.rmtree(LOCAL_CKPT)
# The adapter and tokenizer only: optimizer.pt and the RNG state aren't needed to export.
shutil.copytree(CHECKPOINT_DIR, LOCAL_CKPT,
                ignore=shutil.ignore_patterns("optimizer.pt", "scheduler.pt", "rng_state*"))
_TOKENIZER_FILES = ("tokenizer", "special_tokens", "chat_template", "vocab", "merges",
                    "added_tokens")
_has_tok = any(p.name.startswith("tokenizer") for p in LOCAL_CKPT.iterdir())
if not _has_tok and FINAL_ADAPTER_DIR.is_dir():
    for _f in FINAL_ADAPTER_DIR.iterdir():
        if _f.name.startswith(_TOKENIZER_FILES):
            shutil.copy2(_f, LOCAL_CKPT / _f.name)
    print(f"tokenizer files copied from {FINAL_ADAPTER_DIR}")
print(f"tokenizer in checkpoint: {_has_tok}")

_adapter_cfg = json.loads((LOCAL_CKPT / "adapter_config.json").read_text())
print(f"base model: {_adapter_cfg.get('base_model_name_or_path')}  "
      f"r={_adapter_cfg.get('r')} alpha={_adapter_cfg.get('lora_alpha')}")
print("local copy:", sorted(p.name for p in LOCAL_CKPT.iterdir()))

## 5. Confirm the step from its own trainer state

Prints the train/eval loss history up to this checkpoint, read from the checkpoint's own
`trainer_state.json`, so it is plain which model is being exported. For v2, step 100
should show eval loss of about 0.114.

In [ ]:
_state_path = LOCAL_CKPT / "trainer_state.json"
if _state_path.exists():
    _state = json.loads(_state_path.read_text())
    assert _state.get("global_step") == CHECKPOINT_STEP, (
        f"trainer_state says global_step={_state.get('global_step')}, not {CHECKPOINT_STEP}")
    for _row in _state.get("log_history", []):
        if "eval_loss" in _row:
            print(f"step {_row['step']:>4}  eval_loss {_row['eval_loss']:.4f}")
    print(f"global_step {_state['global_step']} of max_steps {_state.get('max_steps')}")
else:
    print("WARNING: no trainer_state.json in the checkpoint -- step not confirmed")

## 6. Reload base + adapter and smoke-test

Same smoke test as `distill_train.ipynb` section 10: three short held-out prompts must
come back as JSON objects, or nothing is uploaded.

In [ ]:
from unsloth import FastLanguageModel  # first: Unsloth patches transformers on import

from huggingface_hub import hf_hub_download

_val_path = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename="val.jsonl",
                            revision=HF_DATASET_REVISION, token=HF_TOKEN)
with open(_val_path, encoding="utf-8") as _fh:
    val_records = [json.loads(line) for line in _fh if line.strip()]
print(f"{len(val_records)} val records")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(LOCAL_CKPT),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
    token=HF_TOKEN,
)
print(f"reloaded base + adapter from {LOCAL_CKPT}")

FastLanguageModel.for_inference(model)
_short = sorted(val_records, key=lambda r: len(r["messages"][1]["content"]))[:3]
for _r in _short:
    _prompt = tokenizer.apply_chat_template(
        _r["messages"][:-1], tokenize=False, add_generation_prompt=True,
        enable_thinking=False)
    _ids = tokenizer(text=_prompt, return_tensors="pt").to(model.device)
    _out = model.generate(**_ids, max_new_tokens=512, do_sample=False)
    _reply = tokenizer.decode(_out[0][_ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        _ok = isinstance(json.loads(_reply), dict)
    except ValueError:
        _ok = False
    print(f"{_r['prompt_name']:40s} json={_ok} reply={_reply[:120]!r}")
    assert _ok, "the reloaded model did not answer in JSON -- do not upload this export"
print("smoke test passed")

## 7. Push merged safetensors to a new revision

Refuses to push to a revision that already holds weights. That keeps
`opus-distill-2026-10` (the final v2 model) and `main` safe from a mistyped revision. To
re-export the same checkpoint, delete that branch on the Hub first.

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE,
                exist_ok=True)
_branches = {b.name for b in api.list_repo_refs(HF_MODEL_REPO, repo_type="model").branches}
if HF_MODEL_REVISION in _branches:
    _existing = api.list_repo_files(HF_MODEL_REPO, repo_type="model", revision=HF_MODEL_REVISION)
    _weights = [f for f in _existing if f.endswith((".safetensors", ".gguf"))]
    assert not _weights, (
        f"{HF_MODEL_REPO}@{HF_MODEL_REVISION} already holds weights ({_weights[:3]}...) -- "
        "refusing to overwrite; pick another HF_MODEL_REVISION or delete that branch")
api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=HF_MODEL_REVISION,
                  exist_ok=True)

merged_dir = OUT_DIR / "merged-safetensors"
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
_index = json.loads((merged_dir / "model.safetensors.index.json").read_text())
_missing = sorted(set(_index["weight_map"].values())
                  - {p.name for p in merged_dir.glob("*.safetensors")})
assert not _missing, f"merged save is missing shards: {_missing}"

api.upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(merged_dir),
                  revision=HF_MODEL_REVISION,
                  commit_message=f"{RUN_NAME} checkpoint-{CHECKPOINT_STEP}: merged bf16 safetensors")
_pushed = api.list_repo_files(HF_MODEL_REPO, repo_type="model", revision=HF_MODEL_REVISION)
_shards = sorted(f for f in _pushed if f.endswith(".safetensors"))
assert set(_index["weight_map"].values()) <= set(_shards), "not every shard reached the Hub"
print(f"merged safetensors pushed -> {HF_MODEL_REPO}@{HF_MODEL_REVISION}: {_shards}")

## 8. Optional: Q8_0 GGUF (`PUSH_GGUF`)

Skipped by default; vLLM serves the safetensors.

In [ ]:
if PUSH_GGUF:
    gguf_dir = OUT_DIR / "gguf-q8"
    model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q8_0")
    _ggufs = sorted({p for p in OUT_DIR.glob("gguf-q8*/**/*.gguf")})
    _q8 = [p for p in _ggufs if "q8_0" in p.name.lower()] or (_ggufs if len(_ggufs) == 1 else [])
    assert _q8, f"no Q8_0 .gguf found under {OUT_DIR} (found: {[p.name for p in _ggufs]})"
    for _f in _q8:
        api.upload_file(path_or_fileobj=str(_f), path_in_repo=_f.name, repo_id=HF_MODEL_REPO,
                        repo_type="model", revision=HF_MODEL_REVISION,
                        commit_message=f"{RUN_NAME} checkpoint-{CHECKPOINT_STEP}: Q8_0 GGUF")
        print(f"{_f.name} pushed -> {HF_MODEL_REPO}@{HF_MODEL_REVISION}")
else:
    print("PUSH_GGUF is off -- safetensors only")
print(f"DONE: {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

## 9. Next -- outside this notebook

Hand the revision name back. Evaluating it is the same as for the final model:
`MODEL_REVISION=<revision> scripts/vast/provision.sh`, then `scripts/eval_finetune.py
--guided --per-prompt 0`. Next, the faithfulness comparison on the 45 held-out edge chunks.
Last, if it clearly beats v2 there, the end-to-end trial:
`scripts/haiku_extract_ab.py --arms v2 --reuse <routed rows>`.